# Day 1 lab: How a Spark application runs

**Goal:** Explain who does what in a Spark application, trace one action from your code to running tasks, and say how Spark survives a lost executor.

**Time:** about 2 h 30 min. Checklist: 1.1, 1.2, 1.3, 1.5, 1.7, 1.10, 1.11, 1.12.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

## 📘 Code along

Type the code yourself instead of pasting it, run every cell, and keep the Spark UI open. Swap in `spark = get_spark()` from the starter kit wherever an example builds its own session.

### Task 1. Set up PySpark 4.2, Delta 4.4 and the starter kit — 30 min

Install Python 3.12 (`brew install python@3.12`; macOS ships Python 3.9, which PySpark 4.2 does not support) and check `java -version` shows 17 or 21 (Spark 4.2 runs on Java 17, 21 or 25). In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 delta-spark==4.4.1 "pyarrow>=18" "pandas>=2.2,<3" jupyterlab pytest memory-profiler` (PySpark 4.2 does not fully support pandas 3 yet). Download `lab_utils.py` into the folder and run `from lab_utils import *` then `spark = get_spark()`. The kit points executors at your venv's Python and switches off a Spark 4.2.0 setting that can make Python UDFs on Parquet hang or return wrong results (SPARK-60294, fixed in 4.2.1). The first start downloads the Delta and Avro jars (1–2 minutes); later starts take seconds. Open the UI address it prints (usually http://localhost:4040).

[Download lab_utils.py](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py)

In [1]:
# Task 1: your code herefrom
from lab_utils import *
spark = get_spark()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/27 17:45:21 WARN Utils: Your hostname, Usmanovs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.174.192.212 instead (on interface en0)
26/10/27 17:45:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/saidakbar/Documents/innowise/pyspark/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/saidakbar/.ivy2.5.2/cache
The jars for the packages stored in: /Users/saidakbar/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
org.apache.spark#spark-avro_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8052a297-c3c9-4aac-816b-4f3feb8bb423;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unityca

Spark 4.2.0 running, UI at http://10.174.192.212:4040


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 2. Quickstart: DataFrame — 35 min

Work through every section of the official PySpark quickstart in a Jupyter notebook (`jupyter lab`): creating DataFrames from Rows, tuples with a schema string and pandas; viewing and selecting data; a pandas UDF and `mapInPandas`; grouping with `applyInPandas` and `cogroup`; CSV, Parquet and ORC in and out; and SQL. After each `show()`, `collect()` or `toPandas()`, find the new job in the Jobs tab. A real-world bug to fix on the way: on Spark 4.2 the `plus_mean` cell fails with `ArrowInvalid: Float value -3.800000 was truncated converting to int64`, because the function returns floats while `schema=df.schema` says `v1` is a long, and Spark 4 no longer truncates silently (`spark.sql.execution.pandas.convertToArrowArraySafely` is on). Fix it the way a real developer would: declare the true output schema (`v1 double`), not by switching the check off.

[Quickstart: DataFrame](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html) · [Notebook source on GitHub](https://github.com/apache/spark/blob/v4.2.0/python/docs/source/getting_started/quickstart_df.ipynb)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

## ✍️ Exercises

Write these from scratch in the cells below, using the patterns you just typed. Each one ends with a Check you can verify yourself; open the reference solution only after you have a working answer.

### Task 3. Exercise 1: quickstart patterns on your own data — 25 min

Use only what the quickstart showed. (1) Build a 6-person `staff` DataFrame from `Row`s with the schema string `'name string, dept string, salary double, hired date'` (3 departments). (2) Add `salary_k` with `withColumn` and keep people hired after 2020 with `filter`. (3) Average salary per department with `groupby("dept").avg("salary")`. (4) The same through `createOrReplaceTempView("staff")` and `spark.sql`. (5) Centre salaries per department with `applyInPandas`, declaring the output schema correctly. 

**Check:** `from pyspark.testing import assertDataFrameEqual`; `assertDataFrameEqual(api_result, sql_result)` passes, the centred salaries sum to 0 in every department, and jobs appear only for lines with an action. Proves 1.2, 1.11, 1.12.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day01.py)

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 4. Exercise 2: port the official pi.py to DataFrames and spark-submit it — 25 min

Read the Self-Contained Applications section (Python tab) and the official `pi.py`: a script that builds its own SparkSession, uses the RDD API (`parallelize`, `map`, `reduce`) and is launched with `spark-submit`. Write `pi_df.py` that does the same with the DataFrame API: `spark.range(n)`, two `F.rand(seed)` columns for x and y, `F.sum(F.when(x * x + y * y <= 1, 1).otherwise(0))`, print 4 × inside ÷ n, and call `input()` before `spark.stop()` so the UI stays up. Run both with `spark-submit --master "local[4]" pi.py 10` and `spark-submit --master "local[4]" pi_df.py 10`. 

**Check:** both print about 3.14 (within ±0.01 for 1,000,000 points); the RDD job has no entry in the SQL / DataFrame tab, while the DataFrame one shows `HashAggregate` → `Exchange SinglePartition` → `HashAggregate`. Proves 1.1, 1.10, 1.13.

[Exercise 2: port the official pi.py to DataFrames and spark-submit it](https://spark.apache.org/docs/latest/quick-start.html#self-contained-applications) · [pi.py on GitHub](https://github.com/apache/spark/blob/v4.2.0/examples/src/main/python/pi.py) · [Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/pi_df.py)

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

## 🔬 Prove it

Short, focused checks of today's checklist questions in the plan and the Spark UI.

### Task 5. Count jobs, stages and tasks — 20 min

Turn AQE off so the numbers are exact: `spark.conf.set("spark.sql.adaptive.enabled", "false")`. Record jobs, stages and tasks for (a) `orders.count()`, (b) `orders.groupBy("status").count().collect()`, (c) `run(orders.orderBy("amount"))`. Expect 1 job / 2 stages, 1 / 2, and 2 jobs / 3 stages: count adds a single-partition shuffle, and orderBy runs an extra sampling job first. Then set `spark.sql.shuffle.partitions` to 8 and watch the second stage drop from 200 tasks to 8. Proves 1.5, 1.10.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 6. Fail a task, then retry it — 15 min

Write a UDF that raises for one row, e.g. `bad = udf(lambda x: 1 // 0 if x == 7 else x, "long")`, and run `run(spark.range(100).repartition(4).select(bad("id")))`. With master `local[4]` the job fails at the first error (local mode does not retry). Restart Python, start with `get_spark(master="local[4,3]")` (3 attempts per task) and find the 3 failed attempts in the Stages tab. A deterministic bug fails every attempt: retries only help with transient failures such as a lost executor. Proves 1.7.

In [ ]:
# Task 6: your code here


*Task 6 notes: what you predicted, what the plan and the Spark UI showed.*